<a href="https://colab.research.google.com/github/AkhileshSR/AkhileshSR/blob/main/feb2026_2_langgraph_crash_course_part_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# LangGraph Crash Course - Part 2: The Brain

In **Part 1**, we built the **Body** of our agent using pure Python functions. We learned about:
*   **State**: The shared memory.
*   **Nodes**: Functions that do work.
*   **Edges**: How we move from node to node.

Now, in **Part 2**, we will add the **Brain**. We will replace hard-coded logic with **LLMs** (Large Language Models) to make our agents dynamic and intelligent.

We will cover:
1.  **LLM Nodes**: Connecting an LLM to the graph.
2.  **The Router**: Using an LLM to decide which path to take.
3.  **Reflection**: An agent that critiques and improves its own work (Cycles).

In [ ]:
%%capture --no-stderr
%pip install --quiet -U langchain_openai langchain_core langgraph

In [ ]:
import os
import getpass

# If you don't have the key in your environment, input it here
if "OPENAI_API_KEY" not in os.environ:
    os.environ["OPENAI_API_KEY"] = getpass.getpass("Enter your OpenAI API Key: ")

## 1. Simple LLM Node
Let's start by upgrading a simple node to use an LLM.

In [ ]:
from langchain_openai import ChatOpenAI
from typing import TypedDict, Annotated

# Initialize the Model
# We use gpt-4o-mini for economy, or gpt-3.5-turbo for speed/cost
llm = ChatOpenAI(model="gpt-4o-mini")

# 1. State
class State(TypedDict):
    topic: str
    joke: str

# 2. Node
def generate_joke(state: State):
    topic = state["topic"]
    # Provide the task to the LLM
    response = llm.invoke(f"Tell me a short, funny joke about {topic}.")
    return {"joke": response.content}

Now, let's put this node into a simple graph.

In [ ]:
from langgraph.graph import StateGraph, START, END
from IPython.display import Image, display

# 3. Build Graph
builder = StateGraph(State)

builder.add_node("generate_joke", generate_joke)
builder.add_edge(START, "generate_joke")
builder.add_edge("generate_joke", END)

graph = builder.compile()

# Visualize
display(Image(graph.get_graph().draw_mermaid_png()))

# Run
result = graph.invoke({"topic": "AI Agents"})
print(result["joke"])

In [ ]:
# Visualize
display(Image(graph.get_graph().draw_mermaid_png()))

In [ ]:
# Run
result = graph.invoke({"topic": "AI automation"})
print(result["joke"])

Why did the robot go on a diet? 

Because it had too many bytes!


## 2. The Router Pattern (Smart Decisions) 🚦
In Part 1, we used `if "billing" in msg:` to route tickets. This is brittle. If a user says "I was charged improperly", exact keyword matching might fail or get complex.
Let's use an **LLM Router** to classify the intent intelligently.

In [ ]:
from typing import Literal

# 1. State
class RouterState(TypedDict):
    query: str
    category: str
    response: str

# 2. Nodes
# The Router Node - uses the LLM to decide
def llm_router(state: RouterState):
    query = state["query"]

    prompt = f"""
    Classify the following query into one of these categories: 'math', 'writing', 'general'.
    Return ONLY the category name.

    Query: {query}
    """
    response = llm.invoke(prompt)
    category = response.content.strip().lower()

    # Fallback cleanup in case the LLM is chatty
    if "math" in category: category = "math"
    elif "writing" in category: category = "writing"
    else: category = "general"

    return {"category": category}

def math_node(state: RouterState):
    return {"response": "I am the Math Expert. I can solve this using Python Math libraries!"}

def writing_node(state: RouterState):
    return {"response": "I am the Writing Expert. I can write poems and essays!"}

def general_node(state: RouterState):
    return {"response": "I am the General Assistant. I can help with general queries."}

# 3. Conditional Logic
def route_query(state: RouterState) -> Literal["math_node", "writing_node", "general_node"]:
    category = state["category"]
    if category == "math":
        return "math_node"
    elif category == "writing":
        return "writing_node"
    else:
        return "general_node"

In [ ]:
# 4. Build Router Graph
router_builder = StateGraph(RouterState)

router_builder.add_node("router", llm_router)
router_builder.add_node("math_node", math_node)
router_builder.add_node("writing_node", writing_node)
router_builder.add_node("general_node", general_node)

router_builder.add_edge(START, "router")

# Conditional Edge
router_builder.add_conditional_edges(
    "router",
    route_query
)

router_builder.add_edge("math_node", END)
router_builder.add_edge("writing_node", END)
router_builder.add_edge("general_node", END)

router_graph = router_builder.compile()

display(Image(router_graph.get_graph().draw_mermaid_png()))

In [ ]:
# Test the Router
queries = ["What is 55 * 3?", "Write a LinkedIn post about AI", "Hi, how are you?"]

for q in queries:
    res = router_graph.invoke({"query": q})
    print(f"Query: {q:35} -> Category: {res['category']:10} -> Response: {res['response']}")

## 3. The Reflection Pattern (Self-Correction) 🪞
This enables the agent to "Think, Critique, and Improve".
We will use a loop to refine a Tweet until it's perfect.

In [ ]:
from typing import List

# 1. State
class ReflectionState(TypedDict):
    topic: str
    draft: str
    critique: str
    revision_number: int

# 2. Nodes
def generate_draft(state: ReflectionState):
    topic = state["topic"]
    draft = state.get("draft", "")
    critique = state.get("critique", "")
    revision_number = state.get("revision_number", 0)

    # Provide the previous critique if it exists
    if revision_number == 0:
        msg = f"Write a viral tweet about: {topic}"
        print(f"--- Generating Draft (Attempt {revision_number+1}) ---")
    else:
        msg = f"Refine this tweet: '{draft}' based on this critique: '{critique}'"
        print(f"--- Improving Draft (Attempt {revision_number+1}) ---")

    response = llm.invoke(msg)
    return {"draft": response.content, "revision_number": revision_number + 1}

def critique_draft(state: ReflectionState):
    draft = state["draft"]
    print("--- Critiquing Draft ---")
    msg = f"Critique this tweet for virality. Be harsh but constructive. Tweet: '{draft}'"
    response = llm.invoke(msg)
    return {"critique": response.content}

# 3. Conditional Edge (The Loop)
def should_continue(state: ReflectionState) -> Literal["critique_draft", END]:
    # Stop after 2 revisions (3 total drafts)
    if state["revision_number"] > 2:
        return END
    return "critique_draft"

In [ ]:
# 4. Build Reflection Graph
reflection_builder = StateGraph(ReflectionState)

reflection_builder.add_node("generate_draft", generate_draft)
reflection_builder.add_node("critique_draft", critique_draft)

reflection_builder.set_entry_point("generate_draft")

reflection_builder.add_conditional_edges(
    "generate_draft",
    should_continue
)
reflection_builder.add_edge("critique_draft", "generate_draft") # Loop back

reflection_graph = reflection_builder.compile()

display(Image(reflection_graph.get_graph().draw_mermaid_png()))

In [ ]:
# Execute Reflection Agent
print("Generating Viral Tweet...")
result = reflection_graph.invoke({"topic": "Agentic AI", "revision_number": 0})

print("\n=== FINAL TWEET ===")
print(result["draft"])

In the next part (Part 3), we will give these agents **Tools** (like Web Search) and **Memory** so they can have real conversations!